<a href="https://colab.research.google.com/github/Zeenith24/Gen-AI-College/blob/main/GenAIExp6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q transformers==4.46.3 peft==0.13.2 datasets accelerate

In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

In [7]:
data = load_dataset("fancyzhx/ag_news")
train = data["train"].select(range(100))
test = data["test"].select(range(20))

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [8]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize(x):
  return tokenizer(x["text"], padding="max_length", truncation=True, max_length=64)

train = train.map(tokenize, batched=True)
test = test.map(tokenize, batched=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
"distilbert-base-uncased", num_labels=4
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
config = LoraConfig(
r=4,
lora_alpha=8,
lora_dropout=0.1,
target_modules=["q_lin", "v_lin"],
task_type=TaskType.SEQ_CLS
)
model = get_peft_model(model, config)
print("\nLoRA Parameters:")
model.print_trainable_parameters()


LoRA Parameters:
trainable params: 667,396 || all params: 67,623,944 || trainable%: 0.9869


In [11]:
args = TrainingArguments(
  output_dir="result",
  num_train_epochs=1,
  per_device_train_batch_size=16,
  learning_rate=2e-4,
  save_strategy="no",
  report_to="none"
)
trainer = Trainer(
  model=model,
  args=args,
  train_dataset=train,
  processing_class=tokenizer
)
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


TrainOutput(global_step=7, training_loss=1.117842538016183, metrics={'train_runtime': 31.483, 'train_samples_per_second': 3.176, 'train_steps_per_second': 0.222, 'total_flos': 1681529548800.0, 'train_loss': 1.117842538016183, 'epoch': 1.0})

In [12]:
result = trainer.predict(test)
predicted = result.predictions.argmax(axis=1)

In [13]:
classes = {
0: "World",
1: "Sports",
2: "Business",
3: "Technology"
}
# 8. Display predictions
print("\n========== PREDICTIONS ==========")
for i in range(5):
  print("\nText:", test[i]["text"])
  print("Actual:", classes[test[i]["label"]])
  print("Predicted:", classes[predicted[i]])
print("\nPEFT + LoRA Experiment Completed")


========== PREDICTIONS ==========

Text: Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.
Actual: Business
Predicted: Business

Text: The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Canada -- A second\team of rocketeers competing for the  #36;10 million Ansari X Prize, a contest for\privately funded suborbital space flight, has officially announced the first\launch date for its manned rocket.
Actual: Technology
Predicted: Business

Text: Ky. Company Wins Grant to Study Peptides (AP) AP - A company founded by a chemistry researcher at the University of Louisville won a grant to develop a method of producing better peptides, which are short chains of amino acids, the building blocks of proteins.
Actual: Technology
Predicted: Business

Text: Prediction Unit Helps Forecast Wildfires (AP) AP - It's barely dawn when Mike